> 在线讲义：[第 4 章 · 从仿真到真机：域随机化、特权信息与 teacher-student](https://wikieden.github.io/robot-motion-rl/lessons/04-sim2real.html) · 本书《机器人运控与强化学习》

# 第 4 章 · 从仿真到真机：域随机化、特权信息与 teacher-student

配套讲义：仓库里 `robot-motion-rl/notes/lessons/04-sim2real.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（玩具任务是编的，摩擦范围取自 SuperMimic 表 A.3，默认摩擦 1.0 取自附录 F.1）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 只训 1.0 | a = 1/1.0 | `best_action_single(1.0)` | 1.000 |
| 第二步 域随机化 | E[μ]/E[μ²] | `best_action_dr()` | 1.190 |
| 第二步 最坏情况 | 扫 μ ∈ [0.3, 1.2] 取最小 | `worst_case(a)` | −0.490、−0.413 |
| 第三步 真机 μ = 0.4 | −(aμ − 1)² | `reward(a, 0.4)` | −0.360、−0.274、0 |

玩具任务（编的）：动作 a 推一个箱子，效果 = a·μ（μ 是摩擦系数），想让效果正好等于 1，奖励 = −(a·μ − 1)²。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `reward`**：玩具奖励：−(a·μ − 1)²，效果正好是 1 时得 0（最高）。

In [ ]:
def reward(a, mu):
    """玩具奖励：−(a·μ − 1)²，效果正好是 1 时得 0（最高）。"""
    return -((a * mu - 1.0) ** 2)

**函数 `best_action_single`**：只在一个摩擦上训：a = 1/μ_train。

In [ ]:
def best_action_single(mu_train):
    """只在一个摩擦上训：a = 1/μ_train。"""
    return 1.0 / mu_train

**函数 `best_action_dr`**：域随机化：μ ~ U[lo, hi]，最大化平均奖励 → a = E[μ] / E[μ²]。

In [ ]:
def best_action_dr(lo=0.3, hi=1.2):
    """域随机化：μ ~ U[lo, hi]，最大化平均奖励 → a = E[μ] / E[μ²]。"""
    e_mu = (lo + hi) / 2
    e_mu2 = (hi**3 - lo**3) / (3 * (hi - lo))
    return e_mu / e_mu2

**函数 `worst_case`**：在 μ ∈ [lo, hi] 上扫一遍，返回最差的奖励。

In [ ]:
def worst_case(a, lo=0.3, hi=1.2, n=91):
    """在 μ ∈ [lo, hi] 上扫一遍，返回最差的奖励。"""
    return float(min(reward(a, m) for m in np.linspace(lo, hi, n)))

**函数 `best_action_teacher`**：teacher 能看到真实摩擦（特权信息）：a = 1/μ。

In [ ]:
def best_action_teacher(mu_true):
    """teacher 能看到真实摩擦（特权信息）：a = 1/μ。"""
    return 1.0 / mu_true

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 只在 μ=1.0（仿真默认摩擦）上训")
a1 = best_action_single(1.0)
print(f"a = {a1:.3f}")

In [ ]:
print("\n第二步 域随机化 μ~U[0.3, 1.2]")
a2 = best_action_dr()
print(f"a = E[μ]/E[μ²] = {a2:.3f}")

In [ ]:
print("\n第二步（续）最坏情况：μ 在 0.3 和 1.2 两端")
for name, a in [("只训 1.0", a1), ("域随机化", a2)]:
    print(f"{name}: μ=0.3 奖励 {reward(a, 0.3):.3f}，μ=1.2 奖励 {reward(a, 1.2):.3f}，最坏 {worst_case(a):.3f}")

In [ ]:
print("\n第三步 真机 μ=0.4 时三种策略")
for name, a in [("只训 1.0", a1), ("域随机化", a2), ("teacher", best_action_teacher(0.4))]:
    print(f"{name}: a={a:.3f}，效果 {a * 0.4:.3f}，奖励 {reward(a, 0.4):.3f}")

## 3. 代码里才看得到的

- 玩具任务里 DR 的最优动作有解析解；真实训练里没有，PPO 是在抽到的各种参数上平均梯度，效果相同：学到一个对整个分布平均最好的策略。

## 4. 自己改一改

- 把随机化范围缩到 U[0.6, 1.0]，看 DR 的 a 怎么变，μ = 0.4 时奖励变好还是变差。
- 用 `worst_case` 比较几种随机化范围（U[0.3, 1.2]、U[0.5, 1.0]、U[0.2, 1.5]）训出的 DR 策略，看范围越宽最坏情况怎么变。
- 写一个 student：给它一个带噪声的 μ 估计（真值 × (1 + 0.1·噪声)），看平均奖励离 teacher 有多远。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 冰上学会地毯摔，只练一处只会一处；
> 每局换个参数练，处处过得去，买的是稳不是最优；
> 教练知道地滑不滑：给 critic 看，或 teacher 教 student；
> 已知的随机告诉模型，模型就能是确定的。